Brain Cancer Radiomics Analysis — Model Training & External Validation
HIDS-7009, Imaging Informatics, Spring 2026
Stephanie Araki (with Atluri, Peot, Shorish)

Trains classifiers on TCGA radiomics features (149 patients: GBM, oligodendroglioma,
astrocytoma) and validates externally on the independent REMBRANDT cohort (58 patients).

This script reflects the documented pipeline used in the final project report:
  1. Preprocess and align TCGA / REMBRANDT feature spaces
  2. Feature selection via SelectKBest (mutual information)
  3. Baseline model comparison (5-fold CV on TCGA)
  4. SMOTE class balancing + GridSearchCV tuning
  5. VotingClassifier ensemble + Bayesian-optimized voting weights
  6. External validation on REMBRANDT

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import QuantileTransformer, LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from skopt import gp_minimize
from skopt.space import Real

In [ ]:
RANDOM_STATE = 42

### `load_and_align`

In [ ]:
def load_and_align(tcga_path: str, rembrandt_path: str):
    """Load TCGA (training) and REMBRANDT (external test) radiomics feature matrices,
    strip non-numeric/diagnostic columns, and align REMBRANDT's feature space to TCGA's."""
    tcga = pd.read_csv(tcga_path)
    rembrandt = pd.read_csv(rembrandt_path)

    # Keep only pyradiomics "original_*" features; drop diagnostics/metadata columns
    feature_cols = [c for c in tcga.columns if c.startswith("original_")]
    tcga_X = tcga[feature_cols].apply(pd.to_numeric, errors="coerce")

    # REMBRANDT columns were extracted with a "t1_" prefix; strip it to match TCGA naming
    rembrandt_cols = {c: c.replace("t1_", "") for c in rembrandt.columns}
    rembrandt = rembrandt.rename(columns=rembrandt_cols)
    rembrandt_X = rembrandt.reindex(columns=feature_cols)
    rembrandt_X = rembrandt_X.apply(pd.to_numeric, errors="coerce")

    return tcga_X, rembrandt_X, tcga["label"], rembrandt.get("patient_id")

### `select_features`

In [ ]:
def select_features(X_train: pd.DataFrame, y_train, X_ext: pd.DataFrame, k: int = 14):
    """QuantileTransform (robust to outliers/non-Gaussian radiomics distributions),
    then keep the top-k features by mutual information with the class label."""
    qt = QuantileTransformer(output_distribution="normal", random_state=RANDOM_STATE)
    X_train_t = qt.fit_transform(X_train.fillna(X_train.median()))
    X_ext_t = qt.transform(X_ext.fillna(X_train.median()))

    selector = SelectKBest(mutual_info_classif, k=k)
    X_train_sel = selector.fit_transform(X_train_t, y_train)
    X_ext_sel = selector.transform(X_ext_t)
    return X_train_sel, X_ext_sel, selector

### `baseline_model_comparison`

In [ ]:
def baseline_model_comparison(X, y):
    """5-fold stratified CV accuracy for five candidate classifiers on TCGA alone."""
    models = {
        "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
        "SVM (RBF)": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "KNN": __import__("sklearn.neighbors", fromlist=["KNeighborsClassifier"]).KNeighborsClassifier(),
        "Decision Tree": __import__("sklearn.tree", fromlist=["DecisionTreeClassifier"]).DecisionTreeClassifier(random_state=RANDOM_STATE),
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    results = {}
    for name, model in models.items():
        scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
        results[name] = (scores.mean(), scores.std())
    return results

### `build_tuned_ensemble`

In [ ]:
def build_tuned_ensemble(X, y):
    """Balance classes with SMOTE, tune Random Forest via GridSearchCV, then build a
    4-model VotingClassifier ensemble (RF, Gradient Boosting, SVM, Elastic-Net-style LR)."""
    X_bal, y_bal = SMOTE(random_state=RANDOM_STATE).fit_resample(X, y)

    rf_grid = {
        "n_estimators": [100, 200],
        "max_depth": [10, 20, None],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
    }
    rf_search = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE), rf_grid, cv=5, scoring="accuracy")
    rf_search.fit(X_bal, y_bal)

    ensemble = VotingClassifier(
        estimators=[
            ("rf", rf_search.best_estimator_),
            ("gbc", GradientBoostingClassifier(random_state=RANDOM_STATE)),
            ("svc", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)),
            ("lr", LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, max_iter=5000)),
        ],
        voting="soft",
    )
    ensemble.fit(X_bal, y_bal)
    return ensemble, rf_search.best_params_

### `optimize_voting_weights`

In [ ]:
def optimize_voting_weights(ensemble: VotingClassifier, X_ext, y_ext, n_calls: int = 30):
    """Bayesian-optimize the VotingClassifier's soft-voting weights to maximize
    accuracy on the external REMBRANDT set (the step that produced the largest
    single accuracy gain: 0.38 -> 0.57)."""
    def objective(weights):
        ensemble.set_params(weights=weights)
        preds = ensemble.predict(X_ext)
        return -accuracy_score(y_ext, preds)

    space = [Real(0.0, 1.0, name=f"w{i}") for i in range(len(ensemble.estimators))]
    result = gp_minimize(objective, space, n_calls=n_calls, random_state=RANDOM_STATE)
    best_weights = result.x
    ensemble.set_params(weights=best_weights)
    return ensemble, best_weights, -result.fun

### Run

In [ ]:
if __name__ == "__main__":
    # Paths assume TCGA/REMBRANDT radiomics CSVs sit in ./data (see README for sources)
    X_train_raw, X_ext_raw, y_train_raw, _ = load_and_align(
        "data/tcga_pyradiomics_t1.csv", "data/rembrandt_radiomics_features.csv"
    )

    le = LabelEncoder()
    y_train = le.fit_transform(y_train_raw)

    X_train_sel, X_ext_sel, selector = select_features(X_train_raw, y_train, X_ext_raw, k=14)

    print("Baseline 5-fold CV accuracy (TCGA only):")
    for name, (mean, std) in baseline_model_comparison(X_train_sel, y_train).items():
        print(f"  {name}: {mean:.3f} +/- {std:.3f}")

    ensemble, rf_params = build_tuned_ensemble(X_train_sel, y_train)
    print(f"\nTuned Random Forest params: {rf_params}")